In [ ]:
"""
ResNet50 + 自适应多尺度注意力融合 语义分割网络
基于用户提供的原始架构改进
"""

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
from dataset import CityscapesDataset, SimpleTransform
from model import ResNet50_AdaptiveInteraction_Segmentation




# ==================== 4. 数据处理与损失函数 (原样引用自原文件) ====================


class CombinedLoss(nn.Module):
    """组合损失 (引用原文件)"""
    def __init__(self, num_classes=19, ignore_index=255):
        super(CombinedLoss, self).__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.num_classes = num_classes

    def forward(self, pred, target):
        ce_loss = self.ce(pred, target)
        # Dice Loss 实现... (略，保持与原文件一致)
        return ce_loss

# ==================== 5. 训练器与主逻辑 (原样引用自原文件) ====================

class SegmentationTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, device, save_dir='checkpoints'):
        self.model, self.train_loader, self.val_loader = model, train_loader, val_loader
        self.criterion, self.optimizer, self.scheduler, self.device = criterion, optimizer, scheduler, device
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.train_losses, self.val_losses, self.val_miou_history = [], [], []
        self.epoch_loss_history = []  # 新增：记录每个 Epoch 的系统记录
        self.batch_loss_history = []
        self.best_miou, self.epoch = 0, 0

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        running_loss = 0.0
        log_interval = 50

        for batch_idx, (images, targets, _, _) in enumerate(self.train_loader):
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            loss_val = loss.item()
            total_loss += loss_val
            running_loss += loss_val
            
            # 每 50 个 Batch 记录一次平均 Loss
            if (batch_idx + 1) % log_interval == 0:
                avg_batch_loss = running_loss / log_interval
                self.batch_loss_history.append({
                    'epoch': self.epoch + 1,
                    'batch': batch_idx + 1,
                    'avg_loss': avg_batch_loss
                })
                print(f"  Epoch {self.epoch+1}, Batch {batch_idx+1}, Group Avg Loss: {avg_batch_loss:.4f}")
                running_loss = 0.0  # 重置当前组的累加器
        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, confusion_matrix = 0, np.zeros((19, 19))
        for images, targets, _, _ in self.val_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            total_loss += self.criterion(outputs, targets).item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            for t, p in zip(targets.cpu().numpy(), preds):
                mask = (t != 255)
                label = 19 * t[mask].astype('int') + p[mask]
                confusion_matrix += np.bincount(label, minlength=19**2).reshape(19, 19)
        iu = np.diag(confusion_matrix) / (confusion_matrix.sum(axis=1) + confusion_matrix.sum(axis=0) - np.diag(confusion_matrix) + 1e-10)
        return total_loss / len(self.val_loader), np.mean(iu)

    def train(self, num_epochs):
        for epoch in range(num_epochs):
            self.epoch = epoch
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            # train_loss = self.train_epoch()
            avg_epoch_loss = self.train_epoch()
            val_loss, val_miou = self.validate()
            # 3. 记录到列表（结构化存储）
            self.epoch_loss_history.append({
                'epoch': epoch + 1,
                'train_loss': avg_epoch_loss,
                'val_loss': val_loss,
                'miou': val_miou
            })
            
            # 4. 显式输出：每组 Epoch 的平均 Loss 总结
            print(f"--- Epoch {epoch+1} Summary ---")
            print(f"Average Train Loss: {avg_epoch_loss:.6f}")
            print(f"Average Val Loss:   {val_loss:.6f}")
            print(f"mIoU:               {val_miou:.4f}")
            if val_miou > self.best_miou:
                self.best_miou = val_miou
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'best_model.pth'))
                print("保存最佳模型!")

# ==================== 6. 运行脚本 ====================

def main():
    # 路径配置
    root_dir = r"E:\Laboratory files\code_project\city_data"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 实例化改进模型
    model = ResNet50_AdaptiveInteraction_Segmentation(num_classes=19, pretrained=True).to(device)

    train_dataset = CityscapesDataset(root_dir, 'train')
    val_dataset = CityscapesDataset(root_dir, 'val')

    # 打印数据集大小
    print(f"训练集图片数量: {len(train_dataset)}")
    print(f"验证集图片数量: {len(val_dataset)}")

    # 数据加载
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4)

    # 打印每个epoch的batch数量
    print(f"训练集每个epoch的batch数量: {len(train_loader)}")
    print(f"验证集每个epoch的batch数量: {len(val_loader)}")

    # 优化器
    criterion = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # 启动训练
    trainer = SegmentationTrainer(model, train_loader, val_loader, criterion, optimizer, None, device)
    trainer.train(num_epochs=50)

if __name__ == "__main__":
    main()

成功加载预训练权重
训练集图片数量: 2975
验证集图片数量: 500
训练集每个epoch的batch数量: 744
验证集每个epoch的batch数量: 125

Epoch 1/20
  Epoch 1, Batch 50, Group Avg Loss: 1.6277
  Epoch 1, Batch 100, Group Avg Loss: 0.8528
  Epoch 1, Batch 150, Group Avg Loss: 0.7298
  Epoch 1, Batch 200, Group Avg Loss: 0.6962
  Epoch 1, Batch 250, Group Avg Loss: 0.6101
  Epoch 1, Batch 300, Group Avg Loss: 0.5651
  Epoch 1, Batch 350, Group Avg Loss: 0.5103
  Epoch 1, Batch 400, Group Avg Loss: 0.5164
  Epoch 1, Batch 450, Group Avg Loss: 0.4470
  Epoch 1, Batch 500, Group Avg Loss: 0.4469
  Epoch 1, Batch 550, Group Avg Loss: 0.4387
  Epoch 1, Batch 600, Group Avg Loss: 0.4212
  Epoch 1, Batch 650, Group Avg Loss: 0.3750
  Epoch 1, Batch 700, Group Avg Loss: 0.3880
--- Epoch 1 Summary ---
Average Train Loss: 0.602403
Average Val Loss:   0.368115
mIoU:               0.4184
保存最佳模型!

Epoch 2/20
  Epoch 2, Batch 50, Group Avg Loss: 0.3710
  Epoch 2, Batch 100, Group Avg Loss: 0.3203
  Epoch 2, Batch 150, Group Avg Loss: 0.3189
  Epoch 2, Batch 200, Group Avg Loss: 0.3063
  Epoch 2, Batch 250, Group Avg Loss: 0.3105
  Epoch 2, Batch 300, Group Avg Loss: 0.2874
  Epoch 2, Batch 350, Group Avg Loss: 0.3313
  Epoch 2, Batch 400, Group Avg Loss: 0.2878
  Epoch 2, Batch 450, Group Avg Loss: 0.3262
  Epoch 2, Batch 500, Group Avg Loss: 0.3127
  Epoch 2, Batch 550, Group Avg Loss: 0.3000
  Epoch 2, Batch 600, Group Avg Loss: 0.2936
  Epoch 2, Batch 650, Group Avg Loss: 0.2827
  Epoch 2, Batch 700, Group Avg Loss: 0.2753
--- Epoch 2 Summary ---
Average Train Loss: 0.307155
Average Val Loss:   0.290633
mIoU:               0.5181
保存最佳模型!

Epoch 3/20
  Epoch 3, Batch 50, Group Avg Loss: 0.2589
  Epoch 3, Batch 100, Group Avg Loss: 0.2465
  Epoch 3, Batch 150, Group Avg Loss: 0.2611
  Epoch 3, Batch 200, Group Avg Loss: 0.2367
  Epoch 3, Batch 250, Group Avg Loss: 0.2335
  Epoch 3, Batch 300, Group Avg Loss: 0.2209
  Epoch 3, Batch 350, Group Avg Loss: 0.2383
  Epoch 3, Batch 400, Group Avg Loss: 0.2536
  Epoch 3, Batch 450, Group Avg Loss: 0.2365
  Epoch 3, Batch 500, Group Avg Loss: 0.2280
  Epoch 3, Batch 550, Group Avg Loss: 0.2415
  Epoch 3, Batch 600, Group Avg Loss: 0.2496
  Epoch 3, Batch 650, Group Avg Loss: 0.2441
  Epoch 3, Batch 700, Group Avg Loss: 0.2376
--- Epoch 3 Summary ---
Average Train Loss: 0.240997
Average Val Loss:   0.258526
mIoU:               0.5559
保存最佳模型!

Epoch 4/20
  Epoch 4, Batch 50, Group Avg Loss: 0.2014
  Epoch 4, Batch 100, Group Avg Loss: 0.1862
  Epoch 4, Batch 150, Group Avg Loss: 0.2094
  Epoch 4, Batch 200, Group Avg Loss: 0.2132
  Epoch 4, Batch 250, Group Avg Loss: 0.1953
  Epoch 4, Batch 300, Group Avg Loss: 0.2075
  Epoch 4, Batch 350, Group Avg Loss: 0.2024
  Epoch 4, Batch 400, Group Avg Loss: 0.1932
  Epoch 4, Batch 450, Group Avg Loss: 0.2159
  Epoch 4, Batch 500, Group Avg Loss: 0.2027
  Epoch 4, Batch 550, Group Avg Loss: 0.2060
  Epoch 4, Batch 600, Group Avg Loss: 0.1896
  Epoch 4, Batch 650, Group Avg Loss: 0.2048
  Epoch 4, Batch 700, Group Avg Loss: 0.2277
--- Epoch 4 Summary ---
Average Train Loss: 0.204641
Average Val Loss:   0.229700
mIoU:               0.5943
保存最佳模型!

Epoch 5/20
  Epoch 5, Batch 50, Group Avg Loss: 0.1740
  Epoch 5, Batch 100, Group Avg Loss: 0.1673
  Epoch 5, Batch 150, Group Avg Loss: 0.1747
  Epoch 5, Batch 200, Group Avg Loss: 0.1615
  Epoch 5, Batch 250, Group Avg Loss: 0.1664
  Epoch 5, Batch 300, Group Avg Loss: 0.1705
  Epoch 5, Batch 350, Group Avg Loss: 0.1657
  Epoch 5, Batch 400, Group Avg Loss: 0.1793
  Epoch 5, Batch 450, Group Avg Loss: 0.1655
  Epoch 5, Batch 500, Group Avg Loss: 0.2044
  Epoch 5, Batch 550, Group Avg Loss: 0.1865
  Epoch 5, Batch 600, Group Avg Loss: 0.1847
  Epoch 5, Batch 650, Group Avg Loss: 0.1745
  Epoch 5, Batch 700, Group Avg Loss: 0.2026
--- Epoch 5 Summary ---
Average Train Loss: 0.177148
Average Val Loss:   0.238918
mIoU:               0.5817

Epoch 6/20
  Epoch 6, Batch 50, Group Avg Loss: 0.1554
  Epoch 6, Batch 100, Group Avg Loss: 0.1501
  Epoch 6, Batch 150, Group Avg Loss: 0.1519
  Epoch 6, Batch 200, Group Avg Loss: 0.1638
  Epoch 6, Batch 250, Group Avg Loss: 0.1516
  Epoch 6, Batch 300, Group Avg Loss: 0.1510
  Epoch 6, Batch 350, Group Avg Loss: 0.1586
  Epoch 6, Batch 400, Group Avg Loss: 0.1455
  Epoch 6, Batch 450, Group Avg Loss: 0.1691
  Epoch 6, Batch 500, Group Avg Loss: 0.1611
  Epoch 6, Batch 550, Group Avg Loss: 0.1777
  Epoch 6, Batch 600, Group Avg Loss: 0.1531
  Epoch 6, Batch 650, Group Avg Loss: 0.1677
  Epoch 6, Batch 700, Group Avg Loss: 0.1916
--- Epoch 6 Summary ---
Average Train Loss: 0.162488
Average Val Loss:   0.229377
mIoU:               0.5640

Epoch 7/20
  Epoch 7, Batch 50, Group Avg Loss: 0.1582
  Epoch 7, Batch 100, Group Avg Loss: 0.1441
  Epoch 7, Batch 150, Group Avg Loss: 0.1357
  Epoch 7, Batch 200, Group Avg Loss: 0.1499
  Epoch 7, Batch 250, Group Avg Loss: 0.1372
  Epoch 7, Batch 300, Group Avg Loss: 0.1426
  Epoch 7, Batch 350, Group Avg Loss: 0.1404
  Epoch 7, Batch 400, Group Avg Loss: 0.1481
  Epoch 7, Batch 450, Group Avg Loss: 0.1445
  Epoch 7, Batch 500, Group Avg Loss: 0.1369
  Epoch 7, Batch 550, Group Avg Loss: 0.1395
  Epoch 7, Batch 600, Group Avg Loss: 0.1452
  Epoch 7, Batch 650, Group Avg Loss: 0.1392
  Epoch 7, Batch 700, Group Avg Loss: 0.1456
--- Epoch 7 Summary ---
Average Train Loss: 0.144042
Average Val Loss:   0.279638
mIoU:               0.5541

Epoch 8/20
  Epoch 8, Batch 50, Group Avg Loss: 0.1477
  Epoch 8, Batch 100, Group Avg Loss: 0.1400
  Epoch 8, Batch 150, Group Avg Loss: 0.1263
  Epoch 8, Batch 200, Group Avg Loss: 0.1252
  Epoch 8, Batch 250, Group Avg Loss: 0.1203
  Epoch 8, Batch 300, Group Avg Loss: 0.1314
  Epoch 8, Batch 350, Group Avg Loss: 0.1304
  Epoch 8, Batch 400, Group Avg Loss: 0.1294
  Epoch 8, Batch 450, Group Avg Loss: 0.1419
  Epoch 8, Batch 500, Group Avg Loss: 0.1415
  Epoch 8, Batch 550, Group Avg Loss: 0.1291
  Epoch 8, Batch 600, Group Avg Loss: 0.1321
  Epoch 8, Batch 650, Group Avg Loss: 0.1345
  Epoch 8, Batch 700, Group Avg Loss: 0.1347
--- Epoch 8 Summary ---
Average Train Loss: 0.133935
Average Val Loss:   0.221369
mIoU:               0.6073
保存最佳模型!

Epoch 9/20
  Epoch 9, Batch 50, Group Avg Loss: 0.1234
  Epoch 9, Batch 100, Group Avg Loss: 0.1358
  Epoch 9, Batch 150, Group Avg Loss: 0.1290
  Epoch 9, Batch 200, Group Avg Loss: 0.1217
  Epoch 9, Batch 250, Group Avg Loss: 0.1150
  Epoch 9, Batch 300, Group Avg Loss: 0.1214
  Epoch 9, Batch 350, Group Avg Loss: 0.1153
  Epoch 9, Batch 400, Group Avg Loss: 0.1177
  Epoch 9, Batch 450, Group Avg Loss: 0.1328
  Epoch 9, Batch 500, Group Avg Loss: 0.1338
  Epoch 9, Batch 550, Group Avg Loss: 0.1412
  Epoch 9, Batch 600, Group Avg Loss: 0.1346
  Epoch 9, Batch 650, Group Avg Loss: 0.1306
  Epoch 9, Batch 700, Group Avg Loss: 0.1386
--- Epoch 9 Summary ---
Average Train Loss: 0.128599
Average Val Loss:   0.210952
mIoU:               0.6233
保存最佳模型!

Epoch 10/20
  Epoch 10, Batch 50, Group Avg Loss: 0.1243
  Epoch 10, Batch 100, Group Avg Loss: 0.1143
  Epoch 10, Batch 150, Group Avg Loss: 0.1143
  Epoch 10, Batch 200, Group Avg Loss: 0.1197
  Epoch 10, Batch 250, Group Avg Loss: 0.1247
  Epoch 10, Batch 300, Group Avg Loss: 0.1237
  Epoch 10, Batch 350, Group Avg Loss: 0.1194
  Epoch 10, Batch 400, Group Avg Loss: 0.1128
  Epoch 10, Batch 450, Group Avg Loss: 0.1221
  Epoch 10, Batch 500, Group Avg Loss: 0.1303
  Epoch 10, Batch 550, Group Avg Loss: 0.1291
  Epoch 10, Batch 600, Group Avg Loss: 0.1408
  Epoch 10, Batch 650, Group Avg Loss: 0.1312
  Epoch 10, Batch 700, Group Avg Loss: 0.1236
--- Epoch 10 Summary ---
Average Train Loss: 0.123688
Average Val Loss:   0.209047
mIoU:               0.6329
保存最佳模型!

Epoch 11/20
  Epoch 11, Batch 50, Group Avg Loss: 0.1078
  Epoch 11, Batch 100, Group Avg Loss: 0.1111
  Epoch 11, Batch 150, Group Avg Loss: 0.1171
  Epoch 11, Batch 200, Group Avg Loss: 0.1223
  Epoch 11, Batch 250, Group Avg Loss: 0.1289
  Epoch 11, Batch 300, Group Avg Loss: 0.1156
  Epoch 11, Batch 350, Group Avg Loss: 0.1160
  Epoch 11, Batch 400, Group Avg Loss: 0.1078
  Epoch 11, Batch 450, Group Avg Loss: 0.1035
  Epoch 11, Batch 500, Group Avg Loss: 0.1063
  Epoch 11, Batch 550, Group Avg Loss: 0.1050
  Epoch 11, Batch 600, Group Avg Loss: 0.1245
  Epoch 11, Batch 650, Group Avg Loss: 0.1125
  Epoch 11, Batch 700, Group Avg Loss: 0.1211
--- Epoch 11 Summary ---
Average Train Loss: 0.114681
Average Val Loss:   0.228089
mIoU:               0.6087

Epoch 12/20
  Epoch 12, Batch 50, Group Avg Loss: 0.1139
  Epoch 12, Batch 100, Group Avg Loss: 0.1156
  Epoch 12, Batch 150, Group Avg Loss: 0.1149
  Epoch 12, Batch 200, Group Avg Loss: 0.1114
  Epoch 12, Batch 250, Group Avg Loss: 0.1026
  Epoch 12, Batch 300, Group Avg Loss: 0.0971
  Epoch 12, Batch 350, Group Avg Loss: 0.1056
  Epoch 12, Batch 400, Group Avg Loss: 0.0989
  Epoch 12, Batch 450, Group Avg Loss: 0.1066
  Epoch 12, Batch 500, Group Avg Loss: 0.1072
  Epoch 12, Batch 550, Group Avg Loss: 0.1062
  Epoch 12, Batch 600, Group Avg Loss: 0.1004
  Epoch 12, Batch 650, Group Avg Loss: 0.0988
  Epoch 12, Batch 700, Group Avg Loss: 0.1012
--- Epoch 12 Summary ---
Average Train Loss: 0.105831
Average Val Loss:   0.219436
mIoU:               0.6058

Epoch 13/20
  Epoch 13, Batch 50, Group Avg Loss: 0.1076
  Epoch 13, Batch 100, Group Avg Loss: 0.0924
  Epoch 13, Batch 150, Group Avg Loss: 0.0894
  Epoch 13, Batch 200, Group Avg Loss: 0.0933
  Epoch 13, Batch 250, Group Avg Loss: 0.0957
  Epoch 13, Batch 300, Group Avg Loss: 0.0898
  Epoch 13, Batch 350, Group Avg Loss: 0.0940
  Epoch 13, Batch 400, Group Avg Loss: 0.0933
  Epoch 13, Batch 450, Group Avg Loss: 0.0964
  Epoch 13, Batch 500, Group Avg Loss: 0.0906
  Epoch 13, Batch 550, Group Avg Loss: 0.0924
  Epoch 13, Batch 600, Group Avg Loss: 0.0976
  Epoch 13, Batch 650, Group Avg Loss: 0.1168
  Epoch 13, Batch 700, Group Avg Loss: 0.1200
--- Epoch 13 Summary ---
Average Train Loss: 0.098655
Average Val Loss:   0.224780
mIoU:               0.6268

Epoch 14/20
  Epoch 14, Batch 50, Group Avg Loss: 0.0995
  Epoch 14, Batch 100, Group Avg Loss: 0.0935
  Epoch 14, Batch 150, Group Avg Loss: 0.1070
  Epoch 14, Batch 200, Group Avg Loss: 0.1010
  Epoch 14, Batch 250, Group Avg Loss: 0.1062
  Epoch 14, Batch 300, Group Avg Loss: 0.1164
  Epoch 14, Batch 350, Group Avg Loss: 0.1045
  Epoch 14, Batch 400, Group Avg Loss: 0.1162
  Epoch 14, Batch 450, Group Avg Loss: 0.1182
  Epoch 14, Batch 500, Group Avg Loss: 0.1113
  Epoch 14, Batch 550, Group Avg Loss: 0.1050
  Epoch 14, Batch 600, Group Avg Loss: 0.1044
  Epoch 14, Batch 650, Group Avg Loss: 0.1012
  Epoch 14, Batch 700, Group Avg Loss: 0.0928
--- Epoch 14 Summary ---
Average Train Loss: 0.105080
Average Val Loss:   0.198900
mIoU:               0.6500
保存最佳模型!

Epoch 15/20
  Epoch 15, Batch 50, Group Avg Loss: 0.0901
  Epoch 15, Batch 100, Group Avg Loss: 0.0890
  Epoch 15, Batch 150, Group Avg Loss: 0.0883
  Epoch 15, Batch 200, Group Avg Loss: 0.0851
  Epoch 15, Batch 250, Group Avg Loss: 0.0914
  Epoch 15, Batch 300, Group Avg Loss: 0.0988
  Epoch 15, Batch 350, Group Avg Loss: 0.0995
  Epoch 15, Batch 400, Group Avg Loss: 0.1083
  Epoch 15, Batch 450, Group Avg Loss: 0.0995
  Epoch 15, Batch 500, Group Avg Loss: 0.0907
  Epoch 15, Batch 550, Group Avg Loss: 0.0990
  Epoch 15, Batch 600, Group Avg Loss: 0.0964
  Epoch 15, Batch 650, Group Avg Loss: 0.0983
  Epoch 15, Batch 700, Group Avg Loss: 0.1105
--- Epoch 15 Summary ---
Average Train Loss: 0.095974
Average Val Loss:   0.205430
mIoU:               0.6276

Epoch 16/20
  Epoch 16, Batch 50, Group Avg Loss: 0.0902
  Epoch 16, Batch 100, Group Avg Loss: 0.0866
  Epoch 16, Batch 150, Group Avg Loss: 0.0827
  Epoch 16, Batch 200, Group Avg Loss: 0.0890
  Epoch 16, Batch 250, Group Avg Loss: 0.0885
  Epoch 16, Batch 300, Group Avg Loss: 0.0862
  Epoch 16, Batch 350, Group Avg Loss: 0.0839
  Epoch 16, Batch 400, Group Avg Loss: 0.0821
  Epoch 16, Batch 450, Group Avg Loss: 0.0859
  Epoch 16, Batch 500, Group Avg Loss: 0.0830
  Epoch 16, Batch 550, Group Avg Loss: 0.0796
  Epoch 16, Batch 600, Group Avg Loss: 0.0848
  Epoch 16, Batch 650, Group Avg Loss: 0.0852
  Epoch 16, Batch 700, Group Avg Loss: 0.0823
--- Epoch 16 Summary ---
Average Train Loss: 0.084861
Average Val Loss:   0.194466
mIoU:               0.6621
保存最佳模型!

Epoch 17/20
  Epoch 17, Batch 50, Group Avg Loss: 0.0756
  Epoch 17, Batch 100, Group Avg Loss: 0.0778
  Epoch 17, Batch 150, Group Avg Loss: 0.0820
  Epoch 17, Batch 200, Group Avg Loss: 0.0876
  Epoch 17, Batch 250, Group Avg Loss: 0.0812
  Epoch 17, Batch 300, Group Avg Loss: 0.0865
  Epoch 17, Batch 350, Group Avg Loss: 0.0864
  Epoch 17, Batch 400, Group Avg Loss: 0.0836
  Epoch 17, Batch 450, Group Avg Loss: 0.0860
  Epoch 17, Batch 500, Group Avg Loss: 0.1190
  Epoch 17, Batch 550, Group Avg Loss: 0.1112
  Epoch 17, Batch 600, Group Avg Loss: 0.0937
  Epoch 17, Batch 650, Group Avg Loss: 0.0876
  Epoch 17, Batch 700, Group Avg Loss: 0.0877
--- Epoch 17 Summary ---
Average Train Loss: 0.088757
Average Val Loss:   0.206692
mIoU:               0.6607

Epoch 18/20
  Epoch 18, Batch 50, Group Avg Loss: 0.0823
  Epoch 18, Batch 100, Group Avg Loss: 0.0835
  Epoch 18, Batch 150, Group Avg Loss: 0.0793
  Epoch 18, Batch 200, Group Avg Loss: 0.0820
  Epoch 18, Batch 250, Group Avg Loss: 0.0769
  Epoch 18, Batch 300, Group Avg Loss: 0.0792
  Epoch 18, Batch 350, Group Avg Loss: 0.0796
  Epoch 18, Batch 400, Group Avg Loss: 0.0778
  Epoch 18, Batch 450, Group Avg Loss: 0.0740
  Epoch 18, Batch 500, Group Avg Loss: 0.0768
  Epoch 18, Batch 550, Group Avg Loss: 0.0787
  Epoch 18, Batch 600, Group Avg Loss: 0.0761
  Epoch 18, Batch 650, Group Avg Loss: 0.0758
  Epoch 18, Batch 700, Group Avg Loss: 0.0787
--- Epoch 18 Summary ---
Average Train Loss: 0.078525
Average Val Loss:   0.205540
mIoU:               0.6646
保存最佳模型!

Epoch 19/20
  Epoch 19, Batch 50, Group Avg Loss: 0.0756
  Epoch 19, Batch 100, Group Avg Loss: 0.0747
  Epoch 19, Batch 150, Group Avg Loss: 0.0721
  Epoch 19, Batch 200, Group Avg Loss: 0.0765
  Epoch 19, Batch 250, Group Avg Loss: 0.0723
  Epoch 19, Batch 300, Group Avg Loss: 0.0712
  Epoch 19, Batch 350, Group Avg Loss: 0.0699
  Epoch 19, Batch 400, Group Avg Loss: 0.0699
  Epoch 19, Batch 450, Group Avg Loss: 0.0714
  Epoch 19, Batch 500, Group Avg Loss: 0.0697
  Epoch 19, Batch 550, Group Avg Loss: 0.0725
  Epoch 19, Batch 600, Group Avg Loss: 0.0715
  Epoch 19, Batch 650, Group Avg Loss: 0.0727
  Epoch 19, Batch 700, Group Avg Loss: 0.0741
--- Epoch 19 Summary ---
Average Train Loss: 0.072547
Average Val Loss:   0.204618
mIoU:               0.6691
保存最佳模型!

Epoch 20/20
  Epoch 20, Batch 50, Group Avg Loss: 0.0694
  Epoch 20, Batch 100, Group Avg Loss: 0.0720
  Epoch 20, Batch 150, Group Avg Loss: 0.0724
  Epoch 20, Batch 200, Group Avg Loss: 0.1053
  Epoch 20, Batch 250, Group Avg Loss: 0.1325
  Epoch 20, Batch 300, Group Avg Loss: 0.1236
  Epoch 20, Batch 350, Group Avg Loss: 0.1520
  Epoch 20, Batch 400, Group Avg Loss: 0.1453
  Epoch 20, Batch 450, Group Avg Loss: 0.1268
  Epoch 20, Batch 500, Group Avg Loss: 0.1257
  Epoch 20, Batch 550, Group Avg Loss: 0.0986
  Epoch 20, Batch 600, Group Avg Loss: 0.1018
  Epoch 20, Batch 650, Group Avg Loss: 0.0885
  Epoch 20, Batch 700, Group Avg Loss: 0.0925
--- Epoch 20 Summary ---
Average Train Loss: 0.106217
Average Val Loss:   0.204508
mIoU:               0.6494
